# Species development maps — hexagonal harvest/census density

Reads the aggregated GeoPackage (`harvest_hex_<year>` layers) and produces one
multi-panel figure per species × metric (2003–2022) at **publication resolution**,
styled like the reference figure (per-panel frame, lat/lon graticule, national border).

**Key property:** the colour scale is **fixed per species across all years** (breaks are
computed once from all years pooled), so the same colour always means the same density and
change over time is comparable.

Outputs `map_<species>_<metric>.pdf` (and/or `.png`) — vector frame/text with high-DPI
rasterised hex fills, i.e. ArcGIS-layout quality.

*Requires:* `geopandas mapclassify matplotlib numpy pandas` (`pip install ...`).

In [25]:
# ============================ CONFIG — edit these ============================
GPKG        = "hex_aggregated/hexgrid_harvest_8km.gpkg"   # path to your GeoPackage
LAYER_FMT   = "harvest_hex_{year}"          # per-year layer naming
YEARS       = None                          # None = auto-detect all year layers present
OUTDIR      = "maps"
FORMATS     = ["pdf", "png"]                # any of: "pdf", "png"
DPI         = 300                           # 300 publication; 600 for large print
RASTERIZE   = True                          # rasterise hex fills (crisp + small files); False = full vector (huge)
HEX_EDGE    = None                          # hexagon outlines: None = seamless (edge=fill, no visible mesh)
                                            #   or a colour, e.g. "#bbbbbb", to show a visible hex mesh
HEX_EDGE_LW = 0.15                          # outline width when HEX_EDGE is a colour
NCLASS      = 6                             # positive density classes (white 0-class + grey no-data added)
NCOLS       = 5                             # panels per row (5 x 4 = 20 years)

# species key -> (column stem in the gpkg, common name, latin name)
SPECIES = {
    "reddeer":    ("RedDeer",           "Red deer",    "Cervus elaphus"),
    "roedeer":    ("RoeDeer",           "Roe deer",    "Capreolus capreolus"),
    "wildboar":   ("WildBoar",          "Wild boar",   "Sus scrofa"),
    "fallowdeer": ("FallowDeer",        "Fallow deer", "Dama dama"),
    "sikajap":    ("SikaDeerJapaneese", "Sika deer",   "Cervus nippon"),
    "mouflon":    ("Mouflon",           "Mouflon",     "Ovis musimon"),
}
METRICS = {"Bag": "hunting bag", "Spring": "spring census"}   # column prefix -> label

# white 0-class + NCLASS reds (ColorBrewer-style, punchy so low classes stay visible)
FILLS = ["#ffffff","#fdd0bc","#fcae92","#fb7f5b","#ef4634","#c91d13","#8c0f0a"]
GREY  = (0.87, 0.87, 0.87, 1.0)             # no-data colour
# ===========================================================================

In [26]:
import os, re, numpy as np, pandas as pd, geopandas as gpd, mapclassify, warnings, pyogrio
import matplotlib; import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mp
warnings.filterwarnings("ignore")
os.makedirs(OUTDIR, exist_ok=True)
assert len(FILLS) >= NCLASS+1, "need at least NCLASS+1 colours in FILLS"

## 1 · Load layers once, reproject for a lat/lon graticule, cache values

In [27]:
# auto-detect year layers if YEARS not given
_all = [l[0] for l in pyogrio.list_layers(GPKG)]
if YEARS is None:
    yrs = sorted(int(m.group(1)) for l in _all for m in [re.fullmatch(LAYER_FMT.format(year="(\\d{4})"), l)] if m)
else:
    yrs = list(YEARS)
print("year layers:", yrs[0], "…", yrs[-1], f"({len(yrs)} years)")

COLS = [f"{m}_{stem}_Total" for stem,_,_ in SPECIES.values() for m in METRICS]
cache, geom = {}, None
for y in yrs:
    lay = gpd.read_file(GPKG, layer=LAYER_FMT.format(year=y)).to_crs(4326)   # WGS84 for lon/lat ticks
    if geom is None:
        geom = lay[["geometry"]].reset_index(drop=True)          # geometry identical across years
    have = [c for c in COLS if c in lay.columns]
    cache[y] = {c: pd.to_numeric(lay[c], errors="coerce").values for c in have}
COLS = [c for c in COLS if c in cache[yrs[0]]]                    # keep only columns that exist

# --- national border -------------------------------------------------------
# Dissolving the hexagons leaves hairline "sliver" holes wherever adjacent hexagons
# don't share bit-identical vertices. Drawing .boundary would render every sliver as a
# stray black dash inside the country, so drop rings below MIN_HOLE_M2 (a real gap would
# be a whole missing hexagon, ~1e7 m2). Done in the projected CRS so the test is in m^2.
from shapely.geometry import Polygon, MultiPolygon
MIN_HOLE_M2 = 1000.0

def _drop_slivers(poly, min_area=MIN_HOLE_M2):
    keep = [r for r in poly.interiors if Polygon(r).area >= min_area]
    return Polygon(poly.exterior, keep)

_u = geom.to_crs(5514).dissolve().geometry.iloc[0]
_n_before = sum(len(p.interiors) for p in (_u.geoms if _u.geom_type=="MultiPolygon" else [_u]))
_u = (_drop_slivers(_u) if _u.geom_type == "Polygon"
      else MultiPolygon([_drop_slivers(p) for p in _u.geoms]))
_n_after = sum(len(p.interiors) for p in (_u.geoms if _u.geom_type=="MultiPolygon" else [_u]))
border = gpd.GeoSeries([_u], crs=5514).to_crs(4326).iloc[0].boundary
print(f"border: dropped {_n_before-_n_after} sliver ring(s); {_n_after} real hole(s) kept")

minx, miny, maxx, maxy = geom.total_bounds
ASPECT = 1/np.cos(np.radians((miny+maxy)/2))                     # equirectangular aspect

def nice_ticks(lo, hi, target=4):
    span = hi-lo
    for step in (0.5,1,2,2.5,5,10):
        if span/step <= target+1:
            break
    t0 = np.ceil(lo/step)*step
    return np.arange(t0, hi+1e-9, step)
XT, YT = nice_ticks(minx,maxx), nice_ticks(miny,maxy)
print("extent  lon[%.1f, %.1f]  lat[%.1f, %.1f]" % (minx,maxx,miny,maxy))

year layers: 2003 … 2022 (20 years)
border: dropped 215 sliver ring(s); 0 real hole(s) kept
extent  lon[12.0, 18.9]  lat[48.5, 51.1]


## 2 · Fixed colour breaks — pooled over **all** years (per species × metric)
Pooling every year's positive densities and taking quantile breaks is what locks the scale
so a colour means the same density in 2003 and 2022.

In [28]:
def pooled_breaks(stem, met, k=NCLASS):
    col = f"{met}_{stem}_Total"
    if col not in COLS: return None
    v = np.concatenate([cache[y][col][cache[y][col] > 0] for y in yrs])
    v = v[~np.isnan(v)]
    if len(v) < k:  # e.g. an all-null / never-reported species
        print(f"  skip {stem} {met}: only {len(v)} positive values"); return None
    return [float(b) for b in mapclassify.Quantiles(v, k=k).bins]

BREAKS = {}
for key,(stem,_,_) in SPECIES.items():
    for m in METRICS:
        b = pooled_breaks(stem, m)
        if b is not None:
            BREAKS[(key,m)] = b
            print(f"{key:11s} {m:6s} breaks: {[round(x,2) for x in b]}")

reddeer     Bag    breaks: [0.03, 0.09, 0.25, 0.56, 1.19, 25.09]
reddeer     Spring breaks: [0.06, 0.18, 0.4, 0.74, 1.39, 62.8]
roedeer     Bag    breaks: [0.84, 1.16, 1.41, 1.69, 2.07, 10.95]
roedeer     Spring breaks: [2.91, 3.55, 3.97, 4.35, 4.89, 35.46]
wildboar    Bag    breaks: [0.54, 1.05, 1.61, 2.31, 3.44, 106.91]
wildboar    Spring breaks: [0.23, 0.45, 0.7, 0.99, 1.42, 124.71]
fallowdeer  Bag    breaks: [0.02, 0.06, 0.16, 0.39, 1.18, 107.8]
fallowdeer  Spring breaks: [0.07, 0.18, 0.37, 0.73, 1.91, 158.12]
sikajap     Bag    breaks: [0.01, 0.05, 0.21, 0.66, 1.78, 11.78]
sikajap     Spring breaks: [0.06, 0.18, 0.45, 0.96, 2.0, 16.67]
mouflon     Bag    breaks: [0.02, 0.05, 0.11, 0.22, 0.56, 60.45]
mouflon     Spring breaks: [0.08, 0.18, 0.34, 0.59, 1.25, 145.56]


## 3 · Colour mapping + figure builder

In [29]:
def panel_colors(vals, br):
    cmap, norm = ListedColormap(FILLS), BoundaryNorm([0,1e-9]+list(br), len(FILLS))
    out = np.empty((len(vals), 4)); nn = ~np.isnan(vals)
    out[~nn] = GREY; out[nn] = cmap(norm(vals[nn]))
    return out

def make(key, met):
    stem, common, latin = SPECIES[key]
    col = f"{met}_{stem}_Total"
    if (key,met) not in BREAKS: return None
    br = BREAKS[(key,met)]
    nrow = int(np.ceil(len(yrs)/NCOLS))
    fig, axes = plt.subplots(nrow, NCOLS, figsize=(3.1*NCOLS, 2.65*nrow+0.6))
    axes = np.atleast_2d(axes)
    for i, ax in enumerate(axes.ravel()):
        if i >= len(yrs): ax.set_visible(False); continue
        y = yrs[i]; fc = panel_colors(cache[y][col], br)
        geom.plot(ax=ax, color=fc, linewidth=0.25)
        for c in ax.collections:
            # HEX_EDGE=None -> edge colour = fill colour: closes matplotlib's anti-alias seams
            # HEX_EDGE="#..." -> visible hex mesh
            if HEX_EDGE is None: c.set_edgecolor(fc);      c.set_linewidth(0.25)
            else:                c.set_edgecolor(HEX_EDGE); c.set_linewidth(HEX_EDGE_LW)
            c.set_rasterized(RASTERIZE)
        if border.geom_type == "LineString": ax.plot(*border.xy, color="#333", lw=0.6)
        else: [ax.plot(*g.xy, color="#333", lw=0.6) for g in border.geoms]
        ax.set_xlim(minx-0.1, maxx+0.1); ax.set_ylim(miny-0.1, maxy+0.1); ax.set_aspect(ASPECT)
        ax.set_xticks(XT); ax.set_yticks(YT); ax.grid(False)
        row, coln = divmod(i, NCOLS)
        last_row = (row == (len(yrs)-1)//NCOLS)
        ax.tick_params(labelbottom=last_row, labelleft=(coln==0), labelsize=8, length=2, color="#888")
        if coln==0:   ax.set_yticklabels([f"{t:.0f}°N" for t in YT])
        if last_row:  ax.set_xticklabels([f"{t:.0f}°E" for t in XT])
        for s in ax.spines.values(): s.set_edgecolor("#999"); s.set_linewidth(0.6)
        ax.set_title(str(y), fontsize=11, fontweight="bold", color="#222",
                     bbox=dict(boxstyle="round,pad=0.12", fc="#ececec", ec="none"), pad=2)
    fig.suptitle(f"{common} ($\\it{{{latin.replace(' ', chr(92)+' ')}}}$) — {METRICS[met]} (ind./km²)",
                 fontsize=17, y=0.985)
    labs = ["0", f"\u2264{br[0]:.2g}"] + [f"{br[j-1]:.2g}\u2013{br[j]:.2g}" for j in range(1,NCLASS-1)] + [f">{br[NCLASS-2]:.2g}", "no data"]
    hs = [mp.Patch(fc=c, ec="#999", lw=0.3) for c in FILLS[:NCLASS+1]] + [mp.Patch(fc="#dedede", ec="#999", lw=0.3)]
    fig.legend(hs, labs, loc="lower center", ncol=NCLASS+2, fontsize=10, frameon=False,
               bbox_to_anchor=(0.5, 0.022), title="individuals / km²   ·   fixed scale (all years)", title_fontsize=10)
    fig.text(0.5, 0.004, "Hexagonal grid; hexagon value = mean density of overlapping hunting grounds. Grey = no data.",
             ha="center", fontsize=8.5, color="#666")
    plt.subplots_adjust(left=0.035, right=0.99, top=0.93, bottom=0.075, wspace=0.06, hspace=0.02)
    outs = []
    for fmt in FORMATS:
        fn = os.path.join(OUTDIR, f"map_{key}_{met.lower()}.{fmt}")
        fig.savefig(fn, dpi=DPI, bbox_inches="tight"); outs.append(fn)
    plt.close(fig); return outs

## 4 · Render every species × metric

In [30]:
made = []
for key in SPECIES:
    for met in METRICS:
        outs = make(key, met)
        if outs: made += outs; print("built", *outs)
print(f"\n{len(made)} files written to '{OUTDIR}/'")

built maps\map_reddeer_bag.pdf maps\map_reddeer_bag.png
built maps\map_reddeer_spring.pdf maps\map_reddeer_spring.png
built maps\map_roedeer_bag.pdf maps\map_roedeer_bag.png
built maps\map_roedeer_spring.pdf maps\map_roedeer_spring.png
built maps\map_wildboar_bag.pdf maps\map_wildboar_bag.png
built maps\map_wildboar_spring.pdf maps\map_wildboar_spring.png
built maps\map_fallowdeer_bag.pdf maps\map_fallowdeer_bag.png
built maps\map_fallowdeer_spring.pdf maps\map_fallowdeer_spring.png
built maps\map_sikajap_bag.pdf maps\map_sikajap_bag.png
built maps\map_sikajap_spring.pdf maps\map_sikajap_spring.png
built maps\map_mouflon_bag.pdf maps\map_mouflon_bag.png
built maps\map_mouflon_spring.pdf maps\map_mouflon_spring.png

24 files written to 'maps/'


## 5 · (Optional) one large standalone map — single species/year, ArcGIS-style
Adds a scale bar; use for a stand-out single-year figure rather than the grid.

In [31]:
def single_map(key, met, year, dpi=DPI):
    stem, common, latin = SPECIES[key]; col = f"{met}_{stem}_Total"; br = BREAKS[(key,met)]
    fig, ax = plt.subplots(figsize=(9, 6))
    fc = panel_colors(cache[year][col], br)
    geom.plot(ax=ax, color=fc, linewidth=0.3)
    for c in ax.collections:
        if HEX_EDGE is None: c.set_edgecolor(fc);      c.set_linewidth(0.3)
        else:                c.set_edgecolor(HEX_EDGE); c.set_linewidth(max(HEX_EDGE_LW, 0.2))
        c.set_rasterized(RASTERIZE)
    if border.geom_type=="LineString": ax.plot(*border.xy, color="#222", lw=0.9)
    else: [ax.plot(*g.xy, color="#222", lw=0.9) for g in border.geoms]
    ax.set_xlim(minx-0.1,maxx+0.1); ax.set_ylim(miny-0.1,maxy+0.1); ax.set_aspect(ASPECT)
    ax.set_xticks(XT); ax.set_yticks(YT); ax.grid(False)
    ax.set_xticklabels([f"{t:.0f}°E" for t in XT]); ax.set_yticklabels([f"{t:.0f}°N" for t in YT])
    for s in ax.spines.values(): s.set_edgecolor("#666")
    # ~100 km scale bar
    km = 100/ (111*np.cos(np.radians((miny+maxy)/2)))
    x0,y0 = minx+0.2, miny+0.05
    ax.plot([x0,x0+km],[y0,y0], color="k", lw=3, solid_capstyle="butt")
    ax.text(x0+km/2, y0+0.03, "100 km", ha="center", fontsize=8)
    labs=["0",f"\u2264{br[0]:.2g}"]+[f"{br[j-1]:.2g}\u2013{br[j]:.2g}" for j in range(1,NCLASS-1)]+[f">{br[NCLASS-2]:.2g}","no data"]
    hs=[mp.Patch(fc=c,ec="#999",lw=.3) for c in FILLS[:NCLASS+1]]+[mp.Patch(fc="#dedede",ec="#999",lw=.3)]
    ax.legend(hs,labs,loc="lower right",fontsize=8,title="ind./km²",frameon=True)
    ax.set_title(f"{common} (${{\\it {latin.replace(' ',chr(92)+' ')}}}$) — {METRICS[met]} ({year})", fontsize=14)
    fn=os.path.join(OUTDIR,f"single_{key}_{met.lower()}_{year}.pdf")
    fig.savefig(fn,dpi=dpi,bbox_inches="tight"); plt.close(fig); print("built",fn); return fn

# example:
# single_map("wildboar","Bag",2022)

### Tuning tips
- **Resolution:** raise `DPI` to 600 for large print; set `RASTERIZE=False` for fully vector (much larger files).
- **Number of colours:** change `NCLASS` (and add/remove entries in `FILLS`).
- **Different ramp:** replace `FILLS` (keep white first for the 0-class).
- **Hexagon outlines:** `HEX_EDGE = None` gives a seamless surface (recommended for the 20-panel grid — ~1,800 hexes/panel, so a mesh greys it out). Set `HEX_EDGE = "#bbbbbb"` (and tune `HEX_EDGE_LW`) to show individual cells; works best on `single_map()`.
- **Census vs bag:** already both, via the `METRICS` dict.
- **2010–2012** appear sparse because "no harvest" was coded as missing those years — annotate or exclude in the manuscript.